In [ ]:
import pandas as pd 
demandes = pd.read_csv('data/donnees_demandes.csv')
candidats = pd.read_csv('data/candidats_evaluation.csv')

In [19]:
demandes

,id_candidat,cote_r_equivalent,programme_etudes,region_administrative,code_postal_3,revenu_familial_estime,heures_travail_semaine,distance_domicile_campus_km,premiere_generation_universitaire,decision_octroi
0,C008033,25.96,Genie,Cote-Nord,G4R,17017,9,168.0,0,0
1,C004299,26.82,Arts et lettres,Cote-Nord,G5B,66820,22,99.3,0,0
2,C011788,27.37,Genie,Bas-Saint-Laurent,G5N,49223,19,364.1,0,0
3,C001144,21.23,Sciences,Gaspesie-Iles-de-la-Madeleine,G5C,90750,17,174.8,1,0
4,C003322,28.27,Sante,Bas-Saint-Laurent,G5N,36892,9,151.2,0,0
...,...,...,...,...,...,...,...,...,...,...
9995,C006759,28.28,Sciences,Montreal,H4L,75105,6,27.9,0,1
9996,C007105,36.75,Sante,Bas-Saint-Laurent,G0L,51502,16,244.1,0,1
9997,C006790,27.07,Sciences,Montreal,H2X,29940,9,31.7,1,0
9998,C001911,26.13,Genie,Montreal,H1A,56054,10,8.0,0,0


# RandomForestClassifier

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

feats = ["cote_r_equivalent", "heures_travail_semaine", "programme_etudes"]
centres = ["Montreal", "Capitale-Nationale"]

train, test = train_test_split(
    demandes, test_size=0.3, random_state=42, stratify=demandes['decision_octroi']
)

def prep(df):
    X = pd.get_dummies(df[feats], columns=["programme_etudes"])
    y = df["decision_octroi"].astype(int)
    is_centre = df["region_administrative"].isin(centres)
    return X, y, is_centre

X_tr, y_tr, c_tr = prep(train)
X_te, y_te, c_te = prep(test)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)   # align program dummies

# ---- Train on centre applicants only ----
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)
rf.fit(X_tr[c_tr], y_tr[c_tr])

# ---- Evaluate on test, split by group ----
def evaluate(mask, name):
    X, y = X_te[mask], y_te[mask]
    p = rf.predict_proba(X)[:, 1]
    pred = (p >= 0.5).astype(int)
    return {"group": name, "n": len(y),
            "AUC": roc_auc_score(y, p), "accuracy": accuracy_score(y, pred),
            "actual_approval": y.mean(), "predicted_approval": pred.mean()}

out = pd.DataFrame([evaluate(c_te, "centre (test)"),
                    evaluate(~c_te, "remote (test)")]).round(3)
print(out.to_string(index=False))

r = out.set_index("group").loc["remote (test)"]
print(f"\nRemote actually approved:              {r.actual_approval:.1%}")
print(f"Would be approved by centre standard:  {r.predicted_approval:.1%}")
print(f"Estimated remote penalty:              {r.predicted_approval - r.actual_approval:+.1%}")

# ---- Remote test applicants rejected but approvable by centre standard ----
rem = test[~c_te].copy()
rem["pred"] = rf.predict(X_te[~c_te])
flipped = rem[(rem["pred"] == 1) & (rem["decision_octroi"] == 0)]
print(f"\nRejected but approvable: {len(flipped)} of {len(rem)} remote test applicants")
print(flipped[feats].describe().round(2))

        group    n   AUC  accuracy  actual_approval  predicted_approval
centre (test) 1775 0.939     0.863            0.484               0.466
remote (test) 1225 0.933     0.811            0.277               0.429

Remote actually approved:              27.7%
Would be approved by centre standard:  42.9%
Estimated remote penalty:              +15.2%

Rejected but approvable: 209 of 1225 remote test applicants
       cote_r_equivalent  heures_travail_semaine
count             209.00                  209.00
mean               29.04                   12.81
std                 0.99                    3.34
min                27.42                    6.00
25%                28.31                   11.00
50%                28.90                   13.00
75%                29.55                   15.00
max                33.78                   22.00


In [25]:
def prep(df, has_target=True):
    X = pd.get_dummies(df[feats], columns=["programme_etudes"])
    y = df["decision_octroi"].astype(int) if has_target else None
    return X, y

# ---- Unlabeled candidates: keep only the features ----
test_real = candidats[feats].copy()

X_new, _ = prep(test_real, has_target=False)
X_new = X_new.reindex(columns=X_tr.columns, fill_value=0)  # same columns/order as training

predictions = rf.predict(X_new)

soumission = pd.DataFrame({
    'id_candidat': candidats['id_candidat'],
    'decision_octroi': predictions.astype(int),
})
soumission.to_csv('predictions.csv', index=False)

taux = soumission['decision_octroi'].mean()
print(f'{len(soumission)} rows written to predictions.csv')
print(f'Award rate: {taux:.1%}', '- OK' if 0.36 <= taux <= 0.44 else '- OUT OF BUDGET')
soumission.head()

4000 rows written to predictions.csv
Award rate: 46.1% - OUT OF BUDGET


,id_candidat,decision_octroi
0,C011476,0
1,C000471,0
2,C006911,0
3,C007718,0
4,C003531,0


# XGBoost

In [28]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


X_tr_f, X_te_f = X_tr.astype(float), X_te.astype(float)   # bool dummies -> float

models = {
   
    "xgb":    XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8, random_state=0),
   
}

rows = []
for name, m in models.items():
    m.fit(X_tr_f[c_tr], y_tr[c_tr])
    for mask, grp in [(c_te, "centre"), (~c_te, "remote")]:
        X, y = X_te_f[mask], y_te[mask]
        p = m.predict_proba(X)[:, 1]
        rows.append({"model": name, "group": grp, "n": len(y),
                     "AUC": roc_auc_score(y, p),
                     "actual": y.mean(),
                     "pred_mean_prob": p.mean(),          # expected approval rate
                     "pred_thresh_0.5": (p >= 0.5).mean()})
res = pd.DataFrame(rows).round(3)
res["penalty"] = (res["pred_mean_prob"] - res["actual"]).round(3)
print(res.to_string(index=False))

model  group    n   AUC  actual  pred_mean_prob  pred_thresh_0.5  penalty
  xgb centre 1775 0.940   0.484           0.477            0.478   -0.007
  xgb remote 1225 0.931   0.277           0.496            0.512    0.219


In [29]:
import numpy as np, pandas as pd

def prep(df, has_target=True):
    X = pd.get_dummies(df[feats], columns=["programme_etudes"])
    y = df["decision_octroi"].astype(int) if has_target else None
    return X, y

# ---- Unlabeled candidates: keep only the features ----
test_real = candidats[feats].copy()

X_new, _ = prep(test_real, has_target=False)
X_new = X_new.reindex(columns=X_tr.columns, fill_value=0)  # same columns/order as training



TARGET = 0.40          # middle of 36–44%, leaves 4 points of margin each side

def top_k(scores, rate=TARGET):
    k = int(round(rate * len(scores)))
    order = np.argsort(-scores, kind="stable")   # stable: ties broken by row order, reproducible
    pred = np.zeros(len(scores), dtype=int)
    pred[order[:k]] = 1
    return pred

proba_A = m.predict_proba(X_new)[:, 1]
pred_A = top_k(proba_A)

soumission = pd.DataFrame({
    'id_candidat': candidats['id_candidat'],
    'decision_octroi': pred_A.astype(int),
})
soumission.to_csv('predictions_xgb.csv', index=False)

taux = soumission['decision_octroi'].mean()
print(f'{len(soumission)} rows written to predictions.csv')
print(f'Award rate: {taux:.1%}', '- OK' if 0.36 <= taux <= 0.44 else '- OUT OF BUDGET')
soumission.head()

4000 rows written to predictions.csv
Award rate: 40.0% - OK


,id_candidat,decision_octroi
0,C011476,0
1,C000471,0
2,C006911,0
3,C007718,0
4,C003531,0
